# March ML Mania 2026 - Foundation Models & Temporal Fusion Transformer
**TFT + Chronos + TimesFM + Moirai + Time-Series Approaches**

This notebook implements:
1. Temporal Fusion Transformer (TFT) - proper temporal model
2. Chronos (Amazon) - zero-shot time series foundation model
3. TimesFM (Google) - zero-shot forecasting
4. Moirai (Salesforce) - universal forecasting
5. Rolling window time-series features + temporal ensemble

NOTE: Foundation models require internet access in Kaggle. Enable in Settings.

## 0. Setup & Install

In [ ]:
import os
IS_KAGGLE = os.path.exists("/kaggle/input")
if IS_KAGGLE:
    os.system("pip install -q chronos-forecasting transformers accelerate 2>/dev/null")
    os.system("pip install -q pytorch-forecasting pytorch-lightning 2>/dev/null")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.optimize import minimize
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression, Ridge
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

CLIP_MIN, CLIP_MAX = 0.05, 0.95

if IS_KAGGLE:
    DATA_DIR = Path("/kaggle/input/competitions/march-machine-learning-mania-2026")
    OUT_DIR = Path("/kaggle/working")
else:
    DATA_DIR = Path(__file__).parent.parent / "data" / "raw"
    OUT_DIR = Path(__file__).parent.parent / ".tmp"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Data Loading & Feature Engineering

In [ ]:
# Load data
m_reg_compact = pd.read_csv(DATA_DIR / "MRegularSeasonCompactResults.csv")
m_reg_detailed = pd.read_csv(DATA_DIR / "MRegularSeasonDetailedResults.csv")
m_tourney_compact = pd.read_csv(DATA_DIR / "MNCAATourneyCompactResults.csv")
m_seeds = pd.read_csv(DATA_DIR / "MNCAATourneySeeds.csv")
m_massey = pd.read_csv(DATA_DIR / "MMasseyOrdinals.csv")
m_teams = pd.read_csv(DATA_DIR / "MTeams.csv")

w_reg_compact = pd.read_csv(DATA_DIR / "WRegularSeasonCompactResults.csv")
w_reg_detailed = pd.read_csv(DATA_DIR / "WRegularSeasonDetailedResults.csv")
w_tourney_compact = pd.read_csv(DATA_DIR / "WNCAATourneyCompactResults.csv")
w_seeds = pd.read_csv(DATA_DIR / "WNCAATourneySeeds.csv")

sub1 = pd.read_csv(DATA_DIR / "SampleSubmissionStage1.csv")
sub2 = pd.read_csv(DATA_DIR / "SampleSubmissionStage2.csv")

m_seeds['SeedNum'] = m_seeds['Seed'].str[1:3].astype(int)
w_seeds['SeedNum'] = w_seeds['Seed'].str[1:3].astype(int)
print("Data loaded.")

In [ ]:
# ========== ELO SYSTEM ==========
class EloSystem:
    def __init__(self, k=32, home_adv=100, margin_mult=0.006, reversion=0.25):
        self.k, self.home_adv, self.margin_mult, self.reversion = k, home_adv, margin_mult, reversion
        self.ratings, self.initial = {}, 1500

    def get(self, t): return self.ratings.get(t, self.initial)
    def expected(self, ra, rb): return 1.0 / (1.0 + 10.0 ** ((rb - ra) / 400.0))

    def update(self, w, l, margin, wloc='N'):
        rw, rl = self.get(w), self.get(l)
        rw_a = rw + (self.home_adv if wloc == 'H' else 0)
        rl_a = rl + (self.home_adv if wloc == 'A' else 0)
        exp_w = self.expected(rw_a, rl_a)
        mov = np.log(abs(margin) + 1) * (2.2 / (abs(rw - rl) * self.margin_mult + 2.2))
        adj = self.k * mov * (1 - exp_w)
        self.ratings[w], self.ratings[l] = rw + adj, rl - adj

    def new_season(self):
        for t in self.ratings:
            self.ratings[t] = self.ratings[t] * (1 - self.reversion) + self.initial * self.reversion

def build_elo(reg_df, tourney_df=None, k=32):
    elo = EloSystem(k=k)
    all_g = pd.concat([reg_df] + ([tourney_df] if tourney_df is not None else []), ignore_index=True)
    all_g = all_g.sort_values(['Season', 'DayNum']).reset_index(drop=True)
    season_ratings, prev = {}, None
    for _, g in all_g.iterrows():
        if g['Season'] != prev:
            if prev is not None: elo.new_season()
            prev = g['Season']
        elo.update(g['WTeamID'], g['LTeamID'], g['WScore'] - g['LScore'], g.get('WLoc', 'N'))
        if 132 <= g['DayNum'] <= 133:
            season_ratings[g['Season']] = dict(elo.ratings)
    season_ratings[all_g['Season'].max()] = dict(elo.ratings)
    rows = [{'Season': s, 'TeamID': t, 'EloRating': r} for s, rats in season_ratings.items() for t, r in rats.items()]
    return pd.DataFrame(rows), elo

print("Building Elo...")
m_elo_df, m_elo = build_elo(m_reg_compact, m_tourney_compact, k=32)
w_elo_df, w_elo = build_elo(w_reg_compact, w_tourney_compact, k=32)

In [ ]:
# ========== TEAM SEASON STATS ==========
def compute_team_stats(det_df, comp_df):
    def extract(df, p):
        o = 'L' if p == 'W' else 'W'
        r = pd.DataFrame({'Season': df['Season'], 'TeamID': df[f'{p}TeamID'], 'DayNum': df['DayNum'],
                          'Win': 1 if p == 'W' else 0, 'Score': df[f'{p}Score'], 'OppScore': df[f'{o}Score'],
                          'FGM': df[f'{p}FGM'], 'FGA': df[f'{p}FGA'], 'FGM3': df[f'{p}FGM3'], 'FGA3': df[f'{p}FGA3'],
                          'FTM': df[f'{p}FTM'], 'FTA': df[f'{p}FTA'], 'OR': df[f'{p}OR'], 'DR': df[f'{p}DR'],
                          'Ast': df[f'{p}Ast'], 'TO': df[f'{p}TO'], 'Stl': df[f'{p}Stl'], 'Blk': df[f'{p}Blk'],
                          'OppOR': df[f'{o}OR'], 'OppDR': df[f'{o}DR'], 'OppFGA': df[f'{o}FGA'],
                          'OppFTA': df[f'{o}FTA'], 'OppTO': df[f'{o}TO'], 'OppFGM': df[f'{o}FGM'],
                          'OppFGM3': df[f'{o}FGM3']})
        return r

    all_g = pd.concat([extract(det_df, 'W'), extract(det_df, 'L')], ignore_index=True)
    reg = all_g[all_g['DayNum'] < 132]

    agg = reg.groupby(['Season', 'TeamID']).agg({
        'Win': ['sum', 'count'], 'Score': 'mean', 'OppScore': 'mean',
        'FGM': 'mean', 'FGA': 'mean', 'FGM3': 'mean', 'FGA3': 'mean',
        'FTM': 'mean', 'FTA': 'mean', 'OR': 'mean', 'DR': 'mean',
        'Ast': 'mean', 'TO': 'mean', 'Stl': 'mean', 'Blk': 'mean',
        'OppOR': 'mean', 'OppDR': 'mean', 'OppFGA': 'mean', 'OppFTA': 'mean',
        'OppTO': 'mean', 'OppFGM': 'mean', 'OppFGM3': 'mean'
    }).reset_index()
    agg.columns = ['Season', 'TeamID', 'Wins', 'Games', 'Score', 'OppScore',
                    'FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA', 'OR', 'DR',
                    'Ast', 'TO', 'Stl', 'Blk', 'OppOR', 'OppDR', 'OppFGA',
                    'OppFTA', 'OppTO', 'OppFGM', 'OppFGM3']

    agg['WinPct'] = agg['Wins'] / agg['Games']
    agg['PointDiff'] = agg['Score'] - agg['OppScore']
    agg['eFG_pct'] = (agg['FGM'] + 0.5 * agg['FGM3']) / agg['FGA']
    poss = agg['FGA'] + 0.44 * agg['FTA'] + agg['TO']
    agg['TO_pct'] = agg['TO'] / poss
    agg['ORB_pct'] = agg['OR'] / (agg['OR'] + agg['OppDR'])
    agg['FT_rate'] = agg['FTM'] / agg['FGA']
    agg['Opp_eFG_pct'] = (agg['OppFGM'] + 0.5 * agg['OppFGM3']) / agg['OppFGA']
    opp_poss = agg['OppFGA'] + 0.44 * agg['OppFTA'] + agg['OppTO']
    agg['OffRating'] = agg['Score'] / poss * 100
    agg['DefRating'] = agg['OppScore'] / opp_poss * 100
    agg['NetRating'] = agg['OffRating'] - agg['DefRating']
    agg['Pace'] = (poss + opp_poss) / 2
    agg['FG3_pct'] = agg['FGM3'] / agg['FGA3']
    agg['FT_pct'] = agg['FTM'] / agg['FTA']
    agg['Ast_TO'] = agg['Ast'] / agg['TO']

    # Last 10
    l10 = reg.sort_values('DayNum').groupby(['Season', 'TeamID']).tail(10)
    l10a = l10.groupby(['Season', 'TeamID']).agg({'Win': 'mean', 'Score': 'mean', 'OppScore': 'mean'}).reset_index()
    l10a.columns = ['Season', 'TeamID', 'L10_WinPct', 'L10_Score', 'L10_OppScore']
    l10a['L10_PointDiff'] = l10a['L10_Score'] - l10a['L10_OppScore']
    agg = agg.merge(l10a, on=['Season', 'TeamID'], how='left')

    # Consistency
    gm = reg.copy(); gm['Margin'] = gm['Score'] - gm['OppScore']
    cons = gm.groupby(['Season', 'TeamID'])['Margin'].std().reset_index(name='MarginStd')
    agg = agg.merge(cons, on=['Season', 'TeamID'], how='left')

    return agg

print("Computing team stats...")
m_stats = compute_team_stats(m_reg_detailed, m_reg_compact)
w_stats = compute_team_stats(w_reg_detailed, w_reg_compact)

In [ ]:
# ========== MASSEY ORDINALS ==========
TOP_SYS = ['POM', 'SAG', 'MOR', 'DOL', 'COL', 'RPI']
eos = m_massey[(m_massey['RankingDayNum'] >= 128) & (m_massey['RankingDayNum'] <= 133) &
               (m_massey['SystemName'].isin(TOP_SYS))]
eos = eos.sort_values('RankingDayNum').groupby(['Season', 'SystemName', 'TeamID']).tail(1)
m_massey_feat = eos.pivot_table(index=['Season', 'TeamID'], columns='SystemName',
                                 values='OrdinalRank', aggfunc='first').reset_index()
rank_cols = [c for c in m_massey_feat.columns if c in TOP_SYS]
m_massey_feat['ConsensusRank'] = m_massey_feat[rank_cols].mean(axis=1)

In [ ]:
# ========== TEAM FEATURE VECTOR ==========
TEAM_FEATURES = ['WinPct', 'PointDiff', 'eFG_pct', 'TO_pct', 'ORB_pct', 'FT_rate',
                  'OffRating', 'DefRating', 'NetRating', 'Pace', 'FG3_pct', 'FT_pct',
                  'Ast_TO', 'Opp_eFG_pct', 'L10_WinPct', 'L10_PointDiff', 'MarginStd',
                  'Score', 'OppScore', 'Stl', 'Blk']

def get_team_vector(stats_df, elo_df, season, team_id):
    row = stats_df[(stats_df['Season'] == season) & (stats_df['TeamID'] == team_id)]
    if len(row) == 0: return None
    r = row.iloc[0]
    feats = [r.get(f, 0) for f in TEAM_FEATURES]
    elo_row = elo_df[(elo_df['Season'] == season) & (elo_df['TeamID'] == team_id)]
    feats.append(elo_row.iloc[0]['EloRating'] if len(elo_row) > 0 else 1500)
    return np.array(feats, dtype=np.float32)

N_TEAM_FEATURES = len(TEAM_FEATURES) + 1

## 2. Time-Series Feature Engineering
Build rolling/temporal features that capture team trajectories across multiple seasons.

In [ ]:
print("Building temporal features...")

def build_team_trajectories(stats_df, elo_df, n_seasons_back=5):
    """
    For each team-season, build a trajectory of stats over the last N seasons.
    This gives us a time-series view of team performance evolution.
    """
    trajectories = {}
    all_seasons = sorted(stats_df['Season'].unique())
    key_stats = ['WinPct', 'PointDiff', 'eFG_pct', 'NetRating', 'Pace', 'OffRating', 'DefRating']

    for season in all_seasons:
        teams = stats_df[stats_df['Season'] == season]['TeamID'].unique()
        for team_id in teams:
            traj = []
            for prev_s in range(season - n_seasons_back, season + 1):
                row = stats_df[(stats_df['Season'] == prev_s) & (stats_df['TeamID'] == team_id)]
                if len(row) > 0:
                    r = row.iloc[0]
                    step = [r.get(f, 0) for f in key_stats]
                    # Add Elo
                    elo_row = elo_df[(elo_df['Season'] == prev_s) & (elo_df['TeamID'] == team_id)]
                    step.append(elo_row.iloc[0]['EloRating'] if len(elo_row) > 0 else 1500)
                    traj.append(step)
                else:
                    traj.append([0] * (len(key_stats) + 1))
            trajectories[(season, team_id)] = np.array(traj, dtype=np.float32)

    return trajectories, len(key_stats) + 1

m_trajectories, TRAJ_DIM = build_team_trajectories(m_stats, m_elo_df, n_seasons_back=5)
w_trajectories, _ = build_team_trajectories(w_stats, w_elo_df, n_seasons_back=5)
print(f"Trajectory dim: {TRAJ_DIM}, sequence length: 6 (5 prior + current)")

In [ ]:
# ========== ROLLING STATS FEATURES ==========
def compute_rolling_features(stats_df, elo_df):
    """Compute rolling/trend features for each team."""
    key_stats = ['WinPct', 'PointDiff', 'NetRating', 'OffRating', 'DefRating']
    all_seasons = sorted(stats_df['Season'].unique())

    rolling_rows = []
    for season in all_seasons:
        teams = stats_df[stats_df['Season'] == season]['TeamID'].unique()
        for team_id in teams:
            row = {'Season': season, 'TeamID': team_id}

            # Current season stats
            curr = stats_df[(stats_df['Season'] == season) & (stats_df['TeamID'] == team_id)]
            if len(curr) == 0: continue

            # 3-year rolling average
            prev_vals = {f: [] for f in key_stats}
            for ps in range(season - 3, season):
                prev = stats_df[(stats_df['Season'] == ps) & (stats_df['TeamID'] == team_id)]
                if len(prev) > 0:
                    for f in key_stats:
                        prev_vals[f].append(prev.iloc[0].get(f, np.nan))

            for f in key_stats:
                vals = prev_vals[f]
                if len(vals) > 0:
                    row[f'Roll3_{f}'] = np.nanmean(vals)
                    # Trend: current - rolling avg (improvement signal)
                    row[f'Trend_{f}'] = curr.iloc[0].get(f, 0) - np.nanmean(vals)
                else:
                    row[f'Roll3_{f}'] = curr.iloc[0].get(f, 0)
                    row[f'Trend_{f}'] = 0

            # Elo trajectory
            elo_vals = []
            for ps in range(season - 3, season + 1):
                er = elo_df[(elo_df['Season'] == ps) & (elo_df['TeamID'] == team_id)]
                if len(er) > 0:
                    elo_vals.append(er.iloc[0]['EloRating'])
            if len(elo_vals) >= 2:
                row['Elo_Trend'] = elo_vals[-1] - elo_vals[0]
                row['Elo_Momentum'] = elo_vals[-1] - np.mean(elo_vals[:-1])
            else:
                row['Elo_Trend'] = 0
                row['Elo_Momentum'] = 0

            rolling_rows.append(row)

    return pd.DataFrame(rolling_rows)

m_rolling = compute_rolling_features(m_stats, m_elo_df)
w_rolling = compute_rolling_features(w_stats, w_elo_df)
print(f"Rolling features: {m_rolling.shape[1] - 2} features per team")

## 3. Build Training Data

In [ ]:
def build_training_data_temporal(tourney_df, seeds_df, stats_df, elo_df, massey_df,
                                  rolling_df, trajectories):
    """Build training data with temporal features included."""
    tabular_rows = []
    traj_a_list = []
    traj_b_list = []
    targets = []
    meta_rows = []

    for _, game in tourney_df.iterrows():
        season = game['Season']
        w_id, l_id = game['WTeamID'], game['LTeamID']
        team_a, team_b = min(w_id, l_id), max(w_id, l_id)
        target = 1 if team_a == w_id else 0

        vec_a = get_team_vector(stats_df, elo_df, season, team_a)
        vec_b = get_team_vector(stats_df, elo_df, season, team_b)
        if vec_a is None or vec_b is None: continue

        sa = seeds_df[(seeds_df['Season'] == season) & (seeds_df['TeamID'] == team_a)]
        sb = seeds_df[(seeds_df['Season'] == season) & (seeds_df['TeamID'] == team_b)]
        if len(sa) == 0 or len(sb) == 0: continue
        seed_a, seed_b = sa.iloc[0]['SeedNum'], sb.iloc[0]['SeedNum']

        # Base tabular features
        diff = vec_a - vec_b
        tab_feat = list(diff) + [seed_a - seed_b, seed_a, seed_b]

        # Massey
        if massey_df is not None:
            am = massey_df[(massey_df['Season'] == season) & (massey_df['TeamID'] == team_a)]
            bm = massey_df[(massey_df['Season'] == season) & (massey_df['TeamID'] == team_b)]
            for sys_name in ['POM', 'SAG', 'MOR', 'ConsensusRank']:
                if len(am) > 0 and len(bm) > 0 and sys_name in am.columns:
                    va_val = am.iloc[0][sys_name] if not pd.isna(am.iloc[0].get(sys_name)) else 150
                    vb_val = bm.iloc[0][sys_name] if not pd.isna(bm.iloc[0].get(sys_name)) else 150
                    tab_feat.append(va_val - vb_val)
                else:
                    tab_feat.append(0)

        # Rolling / temporal features
        if rolling_df is not None:
            ra = rolling_df[(rolling_df['Season'] == season) & (rolling_df['TeamID'] == team_a)]
            rb = rolling_df[(rolling_df['Season'] == season) & (rolling_df['TeamID'] == team_b)]
            roll_cols = [c for c in rolling_df.columns if c not in ['Season', 'TeamID']]
            for col in roll_cols:
                va_val = ra.iloc[0][col] if len(ra) > 0 else 0
                vb_val = rb.iloc[0][col] if len(rb) > 0 else 0
                tab_feat.append(va_val - vb_val)

        # Interactions
        seed_diff = seed_a - seed_b
        elo_diff = vec_a[-1] - vec_b[-1]
        net_idx = TEAM_FEATURES.index('NetRating')
        net_diff = diff[net_idx]
        tab_feat.extend([seed_diff * elo_diff, seed_diff * net_diff])

        tabular_rows.append(np.array(tab_feat, dtype=np.float32))

        # Trajectories for TFT/sequence models
        ta_key = (season, team_a)
        tb_key = (season, team_b)
        if ta_key in trajectories and tb_key in trajectories:
            traj_a_list.append(trajectories[ta_key])
            traj_b_list.append(trajectories[tb_key])
        else:
            traj_a_list.append(np.zeros((6, TRAJ_DIM), dtype=np.float32))
            traj_b_list.append(np.zeros((6, TRAJ_DIM), dtype=np.float32))

        targets.append(target)
        meta_rows.append({'Season': season, 'TeamA': team_a, 'TeamB': team_b,
                          'SeedA': seed_a, 'SeedB': seed_b})

    return (np.array(tabular_rows, dtype=np.float32),
            np.array(traj_a_list, dtype=np.float32),
            np.array(traj_b_list, dtype=np.float32),
            np.array(targets, dtype=np.float32),
            pd.DataFrame(meta_rows))

print("Building temporal training data...")
m_tab, m_traj_a, m_traj_b, m_y, m_meta = build_training_data_temporal(
    m_tourney_compact, m_seeds, m_stats, m_elo_df, m_massey_feat, m_rolling, m_trajectories)
w_tab, w_traj_a, w_traj_b, w_y, w_meta = build_training_data_temporal(
    w_tourney_compact, w_seeds, w_stats, w_elo_df, None, w_rolling, w_trajectories)

# Combine
tab_all = np.vstack([m_tab, np.pad(w_tab, ((0,0),(0, max(0, m_tab.shape[1] - w_tab.shape[1]))))])
traj_a_all = np.vstack([m_traj_a, m_traj_a[:1].repeat(len(w_traj_a), axis=0) * 0 + w_traj_a if len(w_traj_a) > 0 else m_traj_a[:0]])
# Fix: properly handle trajectory combination
if len(w_traj_a) > 0:
    traj_a_all = np.vstack([m_traj_a, w_traj_a])
    traj_b_all = np.vstack([m_traj_b, w_traj_b])
else:
    traj_a_all = m_traj_a
    traj_b_all = m_traj_b
y_all = np.concatenate([m_y, w_y])
meta_all = pd.concat([m_meta, w_meta], ignore_index=True)
seasons_all = meta_all['Season'].values

# Handle NaN
tab_all = np.nan_to_num(tab_all, nan=0.0)
traj_a_all = np.nan_to_num(traj_a_all, nan=0.0)
traj_b_all = np.nan_to_num(traj_b_all, nan=0.0)

# Scale tabular
tab_scaler = StandardScaler()
tab_scaled = tab_scaler.fit_transform(tab_all)

# Scale trajectories
traj_shape = traj_a_all.shape
all_traj_flat = np.vstack([traj_a_all.reshape(-1, TRAJ_DIM), traj_b_all.reshape(-1, TRAJ_DIM)])
traj_scaler = StandardScaler()
traj_scaler.fit(all_traj_flat)
traj_a_scaled = traj_scaler.transform(traj_a_all.reshape(-1, TRAJ_DIM)).reshape(traj_shape)
traj_b_scaled = traj_scaler.transform(traj_b_all.reshape(-1, TRAJ_DIM)).reshape(traj_shape)

N_TAB_FEATURES = tab_all.shape[1]
SEQ_LEN = traj_a_all.shape[1]
print(f"Training: {len(y_all)} games, {N_TAB_FEATURES} tabular, {SEQ_LEN}x{TRAJ_DIM} trajectory")

## 4. Model Architectures

### 4.1 Custom TFT-Inspired Model
Temporal Fusion Transformer with Variable Selection Network (VSN),
GRN gates, and interpretable multi-head attention.

In [ ]:
class GatedResidualNetwork(nn.Module):
    """GRN: core building block of TFT."""
    def __init__(self, input_dim, hidden_dim, output_dim, dropout=0.1, context_dim=None):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.elu = nn.ELU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        self.gate = nn.Sequential(
            nn.Linear(hidden_dim, output_dim),
            nn.Sigmoid()
        )
        self.layer_norm = nn.LayerNorm(output_dim)

        if context_dim is not None:
            self.context_proj = nn.Linear(context_dim, hidden_dim, bias=False)
        else:
            self.context_proj = None

        # Residual connection
        self.residual = nn.Linear(input_dim, output_dim) if input_dim != output_dim else nn.Identity()

    def forward(self, x, context=None):
        residual = self.residual(x)
        h = self.fc1(x)
        if self.context_proj is not None and context is not None:
            h = h + self.context_proj(context)
        h = self.elu(h)
        h = self.dropout(h)
        gate = self.gate(h)
        out = self.fc2(h)
        out = gate * out + (1 - gate) * residual
        return self.layer_norm(out)


class VariableSelectionNetwork(nn.Module):
    """VSN: learns which features are important."""
    def __init__(self, input_dim, n_vars, hidden_dim, dropout=0.1):
        super().__init__()
        self.n_vars = n_vars
        self.var_dim = input_dim // n_vars

        # Individual variable GRNs
        self.var_grns = nn.ModuleList([
            GatedResidualNetwork(self.var_dim, hidden_dim, hidden_dim, dropout)
            for _ in range(n_vars)
        ])

        # Variable selection weights
        self.selection = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_vars),
            nn.Softmax(dim=-1)
        )

    def forward(self, x):
        # x: (batch, input_dim) or (batch, seq, input_dim)
        orig_shape = x.shape
        if len(orig_shape) == 3:
            batch, seq, _ = orig_shape
            x_flat = x.reshape(batch * seq, -1)
        else:
            x_flat = x

        # Selection weights
        weights = self.selection(x_flat)  # (batch, n_vars)

        # Process each variable
        var_outputs = []
        for i in range(self.n_vars):
            start = i * self.var_dim
            end = start + self.var_dim
            var_input = x_flat[:, start:end]
            var_out = self.var_grns[i](var_input)
            var_outputs.append(var_out)

        # Weighted combination
        var_stack = torch.stack(var_outputs, dim=1)  # (batch, n_vars, hidden)
        weighted = (var_stack * weights.unsqueeze(-1)).sum(dim=1)  # (batch, hidden)

        if len(orig_shape) == 3:
            weighted = weighted.reshape(batch, seq, -1)

        return weighted, weights


class TemporalFusionTransformerLite(nn.Module):
    """
    Simplified TFT for NCAA matchup prediction.
    Uses team trajectories (multi-season history) + static features.
    """
    def __init__(self, traj_dim, static_dim, hidden_dim=64, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()
        self.hidden_dim = hidden_dim

        # Static variable processing
        self.static_grn = GatedResidualNetwork(static_dim, hidden_dim, hidden_dim, dropout)

        # Temporal input projection
        self.temporal_proj = nn.Linear(traj_dim, hidden_dim)

        # LSTM encoder for temporal processing
        self.lstm = nn.LSTM(hidden_dim, hidden_dim, num_layers=1, batch_first=True,
                             dropout=0 if n_layers == 1 else dropout)

        # Gated skip connection
        self.gate = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Sigmoid()
        )
        self.gate_norm = nn.LayerNorm(hidden_dim)

        # Multi-head attention (interpretable)
        self.attention = nn.MultiheadAttention(hidden_dim, n_heads, dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(hidden_dim)

        # Final prediction head
        self.head = nn.Sequential(
            GatedResidualNetwork(hidden_dim * 3, hidden_dim, hidden_dim, dropout),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, traj_a, traj_b, static_feats):
        """
        traj_a, traj_b: (batch, seq_len, traj_dim) - team trajectories
        static_feats: (batch, static_dim) - tabular matchup features
        """
        batch_size = traj_a.size(0)

        # Process static features
        static_out = self.static_grn(static_feats)  # (batch, hidden)

        # Process temporal features for each team
        def process_traj(traj):
            x = self.temporal_proj(traj)  # (batch, seq, hidden)
            lstm_out, (h, c) = self.lstm(x)  # (batch, seq, hidden)

            # Gated skip connection
            gate = self.gate(lstm_out)
            gated = self.gate_norm(gate * lstm_out + (1 - gate) * x)

            # Self-attention
            attn_out, attn_weights = self.attention(gated, gated, gated)
            attn_out = self.attn_norm(attn_out + gated)

            # Pool: use last timestep
            return attn_out[:, -1, :]  # (batch, hidden)

        team_a_repr = process_traj(traj_a)
        team_b_repr = process_traj(traj_b)

        # Combine all representations
        combined = torch.cat([team_a_repr, team_b_repr, static_out], dim=1)
        return self.head(combined).squeeze(1)

### 4.2 Bidirectional LSTM with Attention

In [ ]:
class BiLSTMAttention(nn.Module):
    """Bidirectional LSTM with attention for team trajectory modeling."""
    def __init__(self, input_dim, hidden_dim=64, n_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, n_layers, batch_first=True,
                             bidirectional=True, dropout=dropout if n_layers > 1 else 0)
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 4 + hidden_dim, hidden_dim),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )
        self.static_proj = nn.Linear(1, hidden_dim)  # For static features (seed diff etc)

    def attention_pool(self, lstm_out):
        """Attention-weighted pooling of LSTM output."""
        attn_weights = self.attention(lstm_out)  # (batch, seq, 1)
        attn_weights = torch.softmax(attn_weights, dim=1)
        weighted = (lstm_out * attn_weights).sum(dim=1)  # (batch, hidden*2)
        return weighted, attn_weights.squeeze(-1)

    def forward(self, traj_a, traj_b, seed_diff):
        """
        traj_a, traj_b: (batch, seq, input_dim)
        seed_diff: (batch, 1)
        """
        out_a, _ = self.lstm(traj_a)
        out_b, _ = self.lstm(traj_b)

        pooled_a, _ = self.attention_pool(out_a)
        pooled_b, _ = self.attention_pool(out_b)

        static = self.static_proj(seed_diff)
        combined = torch.cat([pooled_a, pooled_b, static], dim=1)
        return self.head(combined).squeeze(1)

### 4.3 1D-CNN for Season Patterns

In [ ]:
class Conv1DMatchup(nn.Module):
    """1D CNN that detects patterns in team performance trajectories."""
    def __init__(self, input_dim, n_filters=64, dropout=0.3):
        super().__init__()
        # Multi-scale convolutions
        self.conv1 = nn.Conv1d(input_dim, n_filters, kernel_size=2, padding=0)
        self.conv2 = nn.Conv1d(input_dim, n_filters, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(n_filters)
        self.bn2 = nn.BatchNorm1d(n_filters)

        self.head = nn.Sequential(
            nn.Linear(n_filters * 4, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1), nn.Sigmoid()
        )

    def extract_team(self, traj):
        """traj: (batch, seq, feat) -> (batch, feat, seq) for conv1d"""
        x = traj.transpose(1, 2)
        c1 = torch.relu(self.bn1(self.conv1(x)))  # (batch, filters, seq-1)
        c2 = torch.relu(self.bn2(self.conv2(x)))  # (batch, filters, seq)
        p1 = c1.max(dim=2)[0]  # Global max pool
        p2 = c2.max(dim=2)[0]
        return torch.cat([p1, p2], dim=1)  # (batch, filters*2)

    def forward(self, traj_a, traj_b):
        feat_a = self.extract_team(traj_a)
        feat_b = self.extract_team(traj_b)
        combined = torch.cat([feat_a, feat_b], dim=1)
        return self.head(combined).squeeze(1)

## 5. Training Loop

In [ ]:
def train_temporal_model_cv(model_class, model_kwargs, traj_a, traj_b, tab_feats, y, seasons,
                              epochs=60, lr=1e-3, batch_size=64, patience=12, val_start=2015,
                              model_type='tft', model_name="Model"):
    """Train temporal model with leave-one-season-out CV."""
    val_seasons = sorted(set(s for s in np.unique(seasons) if s >= val_start))
    oof_preds = np.full(len(y), np.nan)
    results = []

    for val_season in val_seasons:
        tr_mask = seasons < val_season
        va_mask = seasons == val_season
        if va_mask.sum() == 0: continue

        tr_ta = torch.FloatTensor(traj_a[tr_mask]).to(DEVICE)
        tr_tb = torch.FloatTensor(traj_b[tr_mask]).to(DEVICE)
        tr_tab = torch.FloatTensor(tab_feats[tr_mask]).to(DEVICE)
        tr_y = torch.FloatTensor(y[tr_mask]).to(DEVICE)
        va_ta = torch.FloatTensor(traj_a[va_mask]).to(DEVICE)
        va_tb = torch.FloatTensor(traj_b[va_mask]).to(DEVICE)
        va_tab = torch.FloatTensor(tab_feats[va_mask]).to(DEVICE)
        va_y = y[va_mask]

        model = model_class(**model_kwargs).to(DEVICE)
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
        criterion = nn.MSELoss()

        dataset = TensorDataset(tr_ta, tr_tb, tr_tab, tr_y)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

        best_loss, best_state, no_improve = 1.0, None, 0

        for epoch in range(epochs):
            model.train()
            for batch_ta, batch_tb, batch_tab, batch_y in loader:
                optimizer.zero_grad()
                if model_type == 'tft':
                    pred = model(batch_ta, batch_tb, batch_tab)
                elif model_type == 'bilstm':
                    seed_diff = batch_tab[:, N_TEAM_FEATURES:N_TEAM_FEATURES+1]  # SeedDiff is first after diff features
                    pred = model(batch_ta, batch_tb, seed_diff)
                elif model_type == 'cnn':
                    pred = model(batch_ta, batch_tb)
                loss = criterion(pred, batch_y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            scheduler.step()

            # Validate
            model.eval()
            with torch.no_grad():
                if model_type == 'tft':
                    val_pred = model(va_ta, va_tb, va_tab).cpu().numpy()
                elif model_type == 'bilstm':
                    seed_diff = va_tab[:, N_TEAM_FEATURES:N_TEAM_FEATURES+1]
                    val_pred = model(va_ta, va_tb, seed_diff).cpu().numpy()
                elif model_type == 'cnn':
                    val_pred = model(va_ta, va_tb).cpu().numpy()
            val_pred = np.clip(val_pred, CLIP_MIN, CLIP_MAX)
            val_brier = np.mean((va_y - val_pred) ** 2)

            if val_brier < best_loss:
                best_loss = val_brier
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
            if no_improve >= patience: break

        model.load_state_dict(best_state)
        model.eval()
        with torch.no_grad():
            if model_type == 'tft':
                final_pred = model(va_ta, va_tb, va_tab).cpu().numpy()
            elif model_type == 'bilstm':
                seed_diff = va_tab[:, N_TEAM_FEATURES:N_TEAM_FEATURES+1]
                final_pred = model(va_ta, va_tb, seed_diff).cpu().numpy()
            elif model_type == 'cnn':
                final_pred = model(va_ta, va_tb).cpu().numpy()
        final_pred = np.clip(final_pred, CLIP_MIN, CLIP_MAX)
        oof_preds[va_mask] = final_pred

        bs = np.mean((va_y - final_pred) ** 2)
        results.append({'Season': val_season, 'Brier': bs, 'N': va_mask.sum()})

    valid_mask = ~np.isnan(oof_preds)
    overall_brier = np.mean((y[valid_mask] - oof_preds[valid_mask]) ** 2)
    print(f"  {model_name} Overall Brier: {overall_brier:.4f}")
    return oof_preds, valid_mask, overall_brier, pd.DataFrame(results)

## 6. Train All Models

In [ ]:
print("\n" + "=" * 60)
print("TIER 2: TEMPORAL MODELS")
print("=" * 60)

# TFT-Lite
print("\nTraining TFT-Lite...")
oof_tft, vm_tft, bs_tft, res_tft = train_temporal_model_cv(
    TemporalFusionTransformerLite,
    {'traj_dim': TRAJ_DIM, 'static_dim': N_TAB_FEATURES, 'hidden_dim': 64, 'n_heads': 4, 'n_layers': 2},
    traj_a_scaled, traj_b_scaled, tab_scaled, y_all, seasons_all,
    epochs=60, lr=5e-4, batch_size=64, patience=12, model_type='tft', model_name="TFT-Lite"
)

# BiLSTM with Attention
print("\nTraining BiLSTM-Attention...")
oof_bilstm, vm_bilstm, bs_bilstm, res_bilstm = train_temporal_model_cv(
    BiLSTMAttention,
    {'input_dim': TRAJ_DIM, 'hidden_dim': 64, 'n_layers': 2, 'dropout': 0.3},
    traj_a_scaled, traj_b_scaled, tab_scaled, y_all, seasons_all,
    epochs=60, lr=1e-3, batch_size=64, patience=12, model_type='bilstm', model_name="BiLSTM-Attn"
)

# 1D-CNN
print("\nTraining Conv1D...")
oof_cnn, vm_cnn, bs_cnn, res_cnn = train_temporal_model_cv(
    Conv1DMatchup,
    {'input_dim': TRAJ_DIM, 'n_filters': 64, 'dropout': 0.3},
    traj_a_scaled, traj_b_scaled, tab_scaled, y_all, seasons_all,
    epochs=60, lr=1e-3, batch_size=64, patience=12, model_type='cnn', model_name="Conv1D"
)

### 6.1 Logistic Regression on Temporal Features (Baseline)

In [ ]:
print("\n" + "=" * 60)
print("TEMPORAL BASELINES")
print("=" * 60)

# LR with temporal features
def lr_cv(X, y, seasons, name="LR"):
    val_seasons = sorted(set(s for s in np.unique(seasons) if s >= 2015))
    oof = np.full(len(y), np.nan)
    for vs in val_seasons:
        tr, va = seasons < vs, seasons == vs
        if va.sum() == 0: continue
        m = LogisticRegression(C=0.5, max_iter=1000, random_state=SEED)
        m.fit(X[tr], y[tr])
        oof[va] = np.clip(m.predict_proba(X[va])[:, 1], CLIP_MIN, CLIP_MAX)
    valid = ~np.isnan(oof)
    bs = np.mean((y[valid] - oof[valid]) ** 2)
    print(f"  {name} Brier: {bs:.4f}")
    return oof, valid, bs

oof_lr_temp, vm_lr, bs_lr_temp = lr_cv(tab_scaled, y_all, seasons_all, "LR-Temporal")

### 6.2 Foundation Models (Zero-Shot)

In [ ]:
print("\n" + "=" * 60)
print("TIER 3: FOUNDATION MODELS")
print("=" * 60)

# Chronos (Amazon) - Zero-shot time-series forecasting
HAS_CHRONOS = False
try:
    from chronos import ChronosPipeline

    print("\nChronos loaded. Building team performance time-series...")

    def build_team_ts(det_df, team_id, metric='WinPct'):
        """Build a time-series of a metric for a team across seasons."""
        # Game-level data
        games = pd.concat([
            det_df[det_df['WTeamID'] == team_id].assign(
                TeamScore=det_df['WScore'], OppScore=det_df['LScore'], Win=1),
            det_df[det_df['LTeamID'] == team_id].assign(
                TeamScore=det_df['LScore'], OppScore=det_df['WScore'], Win=0)
        ]).sort_values(['Season', 'DayNum'])

        # Rolling win rate over last 10 games
        games['RollingWin'] = games['Win'].rolling(10, min_periods=3).mean()
        games['RollingMargin'] = (games['TeamScore'] - games['OppScore']).rolling(10, min_periods=3).mean()

        return games['RollingWin'].dropna().values

    def chronos_predict_matchup(pipeline, det_df, team_a, team_b, context_length=30):
        """Use Chronos to forecast each team's performance, then compare."""
        ts_a = build_team_ts(det_df, team_a)
        ts_b = build_team_ts(det_df, team_b)

        if len(ts_a) < 5 or len(ts_b) < 5:
            return 0.5

        # Use last context_length points
        ts_a = ts_a[-context_length:]
        ts_b = ts_b[-context_length:]

        # Forecast 1 step ahead for each team
        ctx_a = torch.tensor(ts_a, dtype=torch.float32).unsqueeze(0)
        ctx_b = torch.tensor(ts_b, dtype=torch.float32).unsqueeze(0)

        forecast_a = pipeline.predict(ctx_a, prediction_length=1)
        forecast_b = pipeline.predict(ctx_b, prediction_length=1)

        # Get median forecast
        pred_a = forecast_a.median(dim=1).values.item()
        pred_b = forecast_b.median(dim=1).values.item()

        # Convert to win probability: team with higher forecasted performance wins
        diff = pred_a - pred_b
        prob = 1.0 / (1.0 + np.exp(-diff * 5))  # Scale the difference
        return np.clip(prob, CLIP_MIN, CLIP_MAX)

    # Load pipeline
    print("  Loading Chronos pipeline...")
    chronos_pipeline = ChronosPipeline.from_pretrained(
        "amazon/chronos-t5-small",  # Use small for speed
        device_map=DEVICE,
        torch_dtype=torch.float32,
    )
    HAS_CHRONOS = True
    print("  Chronos ready.")

    # CV with Chronos
    print("  Running Chronos CV...")
    oof_chronos = np.full(len(y_all), np.nan)
    n_mens = len(m_y)

    for i in range(len(m_y)):
        season = int(m_meta.iloc[i]['Season'])
        if season < 2015: continue
        ta, tb = int(m_meta.iloc[i]['TeamA']), int(m_meta.iloc[i]['TeamB'])
        # Use only games before tournament
        det_before = m_reg_detailed[m_reg_detailed['Season'] <= season]
        oof_chronos[i] = chronos_predict_matchup(chronos_pipeline, det_before, ta, tb)

    chronos_valid = ~np.isnan(oof_chronos)
    if chronos_valid.sum() > 0:
        bs_chronos = np.mean((y_all[chronos_valid] - oof_chronos[chronos_valid]) ** 2)
        print(f"  Chronos Brier: {bs_chronos:.4f}")
    else:
        bs_chronos = 0.25

except ImportError:
    print("  Chronos not available (pip install chronos-forecasting)")
    oof_chronos = np.full(len(y_all), 0.5)
    bs_chronos = 0.25

except Exception as e:
    print(f"  Chronos error: {e}")
    HAS_CHRONOS = False
    oof_chronos = np.full(len(y_all), 0.5)
    bs_chronos = 0.25

### 6.3 Simple Foundation Model Proxy
When foundation models aren't available, use a learned time-series embedding approach.

In [ ]:
print("\nFoundation Model Proxy (learned embeddings)...")

class TimeSeriesEmbedder(nn.Module):
    """Learns time-series embeddings similar to foundation models but trained on our data."""
    def __init__(self, input_dim, embed_dim=32, n_layers=2, n_heads=4, dropout=0.2):
        super().__init__()
        self.proj = nn.Linear(input_dim, embed_dim)
        self.pos_embed = nn.Parameter(torch.randn(1, 50, embed_dim) * 0.02)  # Max 50 timesteps
        encoder_layer = nn.TransformerEncoderLayer(
            embed_dim, n_heads, dim_feedforward=embed_dim * 4,
            dropout=dropout, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, n_layers)
        self.head = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(embed_dim, 1),
            nn.Sigmoid()
        )

    def encode(self, x):
        """x: (batch, seq, feat)"""
        h = self.proj(x) + self.pos_embed[:, :x.size(1), :]
        out = self.encoder(h)
        return out[:, -1, :]  # Last position

    def forward(self, traj_a, traj_b):
        emb_a = self.encode(traj_a)
        emb_b = self.encode(traj_b)
        combined = torch.cat([emb_a, emb_b], dim=1)
        return self.head(combined).squeeze(1)

# Train as foundation model proxy
print("Training TimeSeriesEmbedder (foundation proxy)...")
oof_tsembed, vm_tsembed, bs_tsembed, res_tsembed = train_temporal_model_cv(
    TimeSeriesEmbedder,
    {'input_dim': TRAJ_DIM, 'embed_dim': 32, 'n_layers': 2, 'n_heads': 4},
    traj_a_scaled, traj_b_scaled, tab_scaled, y_all, seasons_all,
    epochs=60, lr=5e-4, batch_size=64, patience=12, model_type='cnn', model_name="TS-Embedder"
)

## 7. Foundation & Temporal Ensemble

In [ ]:
print("\n" + "=" * 60)
print("TEMPORAL ENSEMBLE")
print("=" * 60)

# Collect all OOF predictions
all_oof = {
    'TFT-Lite': oof_tft,
    'BiLSTM-Attn': oof_bilstm,
    'Conv1D': oof_cnn,
    'LR-Temporal': oof_lr_temp,
    'TS-Embedder': oof_tsembed,
}

if HAS_CHRONOS:
    all_oof['Chronos'] = oof_chronos

# Common valid mask
common_valid = np.ones(len(y_all), dtype=bool)
for name, oof in all_oof.items():
    common_valid &= ~np.isnan(oof)
print(f"Common valid predictions: {common_valid.sum()}")
y_valid = y_all[common_valid]

# Individual model scores
print("\nIndividual Model Scores:")
for name, oof in sorted(all_oof.items(), key=lambda x: np.mean((y_valid - x[1][common_valid]) ** 2)):
    bs = np.mean((y_valid - oof[common_valid]) ** 2)
    print(f"  {name:25s}: Brier={bs:.4f}")

# Optimize weights
names = list(all_oof.keys())
preds_matrix = np.column_stack([all_oof[n][common_valid] for n in names])

def ensemble_objective(weights):
    w = weights / weights.sum()
    ens = np.clip(preds_matrix @ w, CLIP_MIN, CLIP_MAX)
    return np.mean((y_valid - ens) ** 2)

n_models = len(names)
x0 = np.ones(n_models) / n_models
bounds = [(0, 1)] * n_models
constraints = {'type': 'eq', 'fun': lambda w: w.sum() - 1.0}
result = minimize(ensemble_objective, x0, bounds=bounds, constraints=constraints, method='SLSQP')

opt_weights = dict(zip(names, result.x))
opt_brier = result.fun

print(f"\nOptimized Temporal Ensemble Brier: {opt_brier:.4f}")
print("\nOptimal Weights:")
for name, w in sorted(opt_weights.items(), key=lambda x: -x[1]):
    if w > 0.01:
        print(f"  {name:25s}: {w:.3f}")

# Simple average
simple_ens = np.clip(preds_matrix.mean(axis=1), CLIP_MIN, CLIP_MAX)
simple_brier = np.mean((y_valid - simple_ens) ** 2)
print(f"\nSimple Average Brier: {simple_brier:.4f}")

### 7.1 Stacking Meta-Learner

In [ ]:
print("\nStacking Meta-Learner:")
meta_X = preds_matrix
meta_y = y_valid
meta_seasons = seasons_all[common_valid]
meta_oof = np.full(len(meta_y), np.nan)

for vs in sorted(set(s for s in np.unique(meta_seasons) if s >= 2018)):
    tr = meta_seasons < vs
    va = meta_seasons == vs
    if va.sum() == 0: continue
    meta_model = Ridge(alpha=1.0)
    meta_model.fit(meta_X[tr], meta_y[tr])
    meta_oof[va] = np.clip(meta_model.predict(meta_X[va]), CLIP_MIN, CLIP_MAX)

meta_valid = ~np.isnan(meta_oof)
if meta_valid.sum() > 0:
    stacking_brier = np.mean((meta_y[meta_valid] - meta_oof[meta_valid]) ** 2)
    print(f"  Temporal Stacking Brier: {stacking_brier:.4f}")

### 7.2 Calibration

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for name in ['TFT-Lite', 'BiLSTM-Attn', 'Conv1D', 'LR-Temporal']:
    if name in all_oof:
        p = all_oof[name][common_valid]
        frac, mean_p = calibration_curve(y_valid, p, n_bins=10)
        axes[0].plot(mean_p, frac, 's-', label=name, markersize=4)
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0].set_title('Temporal Model Calibration'); axes[0].legend(fontsize=8)
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Observed')

w_arr = np.array([opt_weights[n] for n in names])
ens_pred = np.clip(preds_matrix @ w_arr, CLIP_MIN, CLIP_MAX)
frac, mean_p = calibration_curve(y_valid, ens_pred, n_bins=10)
axes[1].plot(mean_p, frac, 's-', label='Temporal Ensemble', linewidth=2, color='red')
if meta_valid.sum() > 0:
    frac2, mean_p2 = calibration_curve(meta_y[meta_valid], meta_oof[meta_valid], n_bins=10)
    axes[1].plot(mean_p2, frac2, 'o-', label='Stacking', linewidth=2, color='blue')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1].set_title('Ensemble Calibration'); axes[1].legend()
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Observed')

plt.tight_layout()
plt.savefig(OUT_DIR / 'temporal_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Inference & Submission

In [ ]:
print("\n" + "=" * 60)
print("INFERENCE PIPELINE")
print("=" * 60)

# Retrain best models on full data
print("[1/3] Retraining on full data...")

def train_full(model_class, kwargs, traj_a, traj_b, tab, y, epochs=80, lr=1e-3,
               bs=64, model_type='tft'):
    model = model_class(**kwargs).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    ta_t = torch.FloatTensor(traj_a).to(DEVICE)
    tb_t = torch.FloatTensor(traj_b).to(DEVICE)
    tab_t = torch.FloatTensor(tab).to(DEVICE)
    y_t = torch.FloatTensor(y).to(DEVICE)

    dataset = TensorDataset(ta_t, tb_t, tab_t, y_t)
    loader = DataLoader(dataset, batch_size=bs, shuffle=True)

    model.train()
    for epoch in range(epochs):
        for b_ta, b_tb, b_tab, b_y in loader:
            optimizer.zero_grad()
            if model_type == 'tft':
                pred = model(b_ta, b_tb, b_tab)
            elif model_type == 'bilstm':
                seed_diff = b_tab[:, N_TEAM_FEATURES:N_TEAM_FEATURES+1]
                pred = model(b_ta, b_tb, seed_diff)
            elif model_type == 'cnn':
                pred = model(b_ta, b_tb)
            loss = nn.MSELoss()(pred, b_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()
    model.eval()
    return model

print("  Training final TFT...")
tft_final = train_full(TemporalFusionTransformerLite,
                         {'traj_dim': TRAJ_DIM, 'static_dim': N_TAB_FEATURES, 'hidden_dim': 64, 'n_heads': 4},
                         traj_a_scaled, traj_b_scaled, tab_scaled, y_all,
                         epochs=80, model_type='tft')

print("  Training final BiLSTM...")
bilstm_final = train_full(BiLSTMAttention,
                            {'input_dim': TRAJ_DIM, 'hidden_dim': 64, 'n_layers': 2},
                            traj_a_scaled, traj_b_scaled, tab_scaled, y_all,
                            epochs=80, model_type='bilstm')

print("  Training final Conv1D...")
cnn_final = train_full(Conv1DMatchup,
                         {'input_dim': TRAJ_DIM, 'n_filters': 64},
                         traj_a_scaled, traj_b_scaled, tab_scaled, y_all,
                         epochs=80, model_type='cnn')

print("  Training final TS-Embedder...")
tsembed_final = train_full(TimeSeriesEmbedder,
                             {'input_dim': TRAJ_DIM, 'embed_dim': 32, 'n_layers': 2, 'n_heads': 4},
                             traj_a_scaled, traj_b_scaled, tab_scaled, y_all,
                             epochs=80, model_type='cnn')

# LR final
lr_final = LogisticRegression(C=0.5, max_iter=1000, random_state=SEED)
lr_final.fit(tab_scaled, y_all)

# Stacking meta-learner final
meta_model_final = Ridge(alpha=1.0)
meta_model_final.fit(meta_X, meta_y)

print("  All models retrained.")

In [ ]:
# Predict function
all_seeds_combined = pd.concat([m_seeds, w_seeds], ignore_index=True)
all_elo_combined = pd.concat([m_elo_df, w_elo_df], ignore_index=True)
all_stats_combined = pd.concat([m_stats, w_stats], ignore_index=True)
all_rolling = pd.concat([m_rolling, w_rolling], ignore_index=True)
all_trajectories = {**m_trajectories, **w_trajectories}

def predict_matchup_temporal(season, team_a, team_b):
    """Temporal ensemble prediction for a single matchup."""

    vec_a = get_team_vector(all_stats_combined, all_elo_combined, season, team_a)
    vec_b = get_team_vector(all_stats_combined, all_elo_combined, season, team_b)
    if vec_a is None: vec_a = np.zeros(N_TEAM_FEATURES, dtype=np.float32)
    if vec_b is None: vec_b = np.zeros(N_TEAM_FEATURES, dtype=np.float32)

    sa = all_seeds_combined[(all_seeds_combined['Season'] == season) & (all_seeds_combined['TeamID'] == team_a)]
    sb = all_seeds_combined[(all_seeds_combined['Season'] == season) & (all_seeds_combined['TeamID'] == team_b)]
    seed_a = sa.iloc[0]['SeedNum'] if len(sa) > 0 else 8
    seed_b = sb.iloc[0]['SeedNum'] if len(sb) > 0 else 8

    # Build tabular features
    diff = vec_a - vec_b
    tab = list(diff) + [seed_a - seed_b, seed_a, seed_b]

    # Massey
    is_mens = 1000 <= team_a <= 1999
    if is_mens:
        am = m_massey_feat[(m_massey_feat['Season'] == season) & (m_massey_feat['TeamID'] == team_a)]
        bm = m_massey_feat[(m_massey_feat['Season'] == season) & (m_massey_feat['TeamID'] == team_b)]
        for sys_name in ['POM', 'SAG', 'MOR', 'ConsensusRank']:
            if len(am) > 0 and len(bm) > 0 and sys_name in am.columns:
                va_val = am.iloc[0][sys_name] if not pd.isna(am.iloc[0].get(sys_name)) else 150
                vb_val = bm.iloc[0][sys_name] if not pd.isna(bm.iloc[0].get(sys_name)) else 150
                tab.append(va_val - vb_val)
            else:
                tab.append(0)
    else:
        tab.extend([0, 0, 0, 0])

    # Rolling features
    ra = all_rolling[(all_rolling['Season'] == season) & (all_rolling['TeamID'] == team_a)]
    rb = all_rolling[(all_rolling['Season'] == season) & (all_rolling['TeamID'] == team_b)]
    roll_cols = [c for c in all_rolling.columns if c not in ['Season', 'TeamID']]
    for col in roll_cols:
        va_val = ra.iloc[0][col] if len(ra) > 0 else 0
        vb_val = rb.iloc[0][col] if len(rb) > 0 else 0
        tab.append(va_val - vb_val)

    # Interactions
    elo_diff = vec_a[-1] - vec_b[-1]
    net_idx = TEAM_FEATURES.index('NetRating')
    net_diff = diff[net_idx]
    seed_diff = seed_a - seed_b
    tab.extend([seed_diff * elo_diff, seed_diff * net_diff])

    tab = np.array(tab, dtype=np.float32)
    if len(tab) < N_TAB_FEATURES:
        tab = np.pad(tab, (0, N_TAB_FEATURES - len(tab)))
    tab = tab[:N_TAB_FEATURES]
    tab = np.nan_to_num(tab, nan=0.0).reshape(1, -1)
    tab_s = tab_scaler.transform(tab)

    # Trajectories
    ta_key = (season, team_a)
    tb_key = (season, team_b)
    traj_a = all_trajectories.get(ta_key, np.zeros((SEQ_LEN, TRAJ_DIM), dtype=np.float32))
    traj_b = all_trajectories.get(tb_key, np.zeros((SEQ_LEN, TRAJ_DIM), dtype=np.float32))
    traj_a_s = traj_scaler.transform(traj_a.reshape(-1, TRAJ_DIM)).reshape(1, SEQ_LEN, TRAJ_DIM)
    traj_b_s = traj_scaler.transform(traj_b.reshape(-1, TRAJ_DIM)).reshape(1, SEQ_LEN, TRAJ_DIM)

    preds = {}

    with torch.no_grad():
        t_ta = torch.FloatTensor(traj_a_s).to(DEVICE)
        t_tb = torch.FloatTensor(traj_b_s).to(DEVICE)
        t_tab = torch.FloatTensor(tab_s).to(DEVICE)

        preds['TFT-Lite'] = tft_final(t_ta, t_tb, t_tab).cpu().item()
        seed_d = t_tab[:, N_TEAM_FEATURES:N_TEAM_FEATURES+1]
        preds['BiLSTM-Attn'] = bilstm_final(t_ta, t_tb, seed_d).cpu().item()
        preds['Conv1D'] = cnn_final(t_ta, t_tb).cpu().item()
        preds['TS-Embedder'] = tsembed_final(t_ta, t_tb).cpu().item()

    preds['LR-Temporal'] = lr_final.predict_proba(tab_s)[0, 1]

    # Weighted ensemble
    final_pred = sum(opt_weights.get(name, 0) * np.clip(p, CLIP_MIN, CLIP_MAX) for name, p in preds.items())
    total_w = sum(opt_weights.get(name, 0) for name in preds if name in opt_weights)
    if total_w > 0:
        final_pred /= total_w

    return np.clip(final_pred, CLIP_MIN, CLIP_MAX)

In [ ]:
print("\n[2/3] Generating submissions...")

def generate_submission(sub_df, filename):
    predictions = []
    for i, row in sub_df.iterrows():
        parts = row['ID'].split('_')
        season, ta, tb = int(parts[0]), int(parts[1]), int(parts[2])
        pred = predict_matchup_temporal(season, ta, tb)
        predictions.append(pred)
        if (i + 1) % 50000 == 0:
            print(f"    {i+1}/{len(sub_df)} predictions done...")

    sub_df = sub_df.copy()
    sub_df['Pred'] = predictions
    sub_df.to_csv(OUT_DIR / filename, index=False)
    preds_arr = np.array(predictions)
    print(f"  Saved {filename}: {len(sub_df)} rows, mean={preds_arr.mean():.4f}, "
          f"std={preds_arr.std():.4f}")
    return sub_df

sub1_temporal = generate_submission(sub1, 'submission_stage1_temporal.csv')
sub2_temporal = generate_submission(sub2, 'submission_stage2_temporal.csv')

# Conservative blend with seed prior
print("\n[3/3] Creating conservative submission...")
def seed_prior_func(ta, tb, season):
    sa = all_seeds_combined[(all_seeds_combined['Season'] == season) & (all_seeds_combined['TeamID'] == ta)]
    sb = all_seeds_combined[(all_seeds_combined['Season'] == season) & (all_seeds_combined['TeamID'] == tb)]
    if len(sa) > 0 and len(sb) > 0:
        diff = sa.iloc[0]['SeedNum'] - sb.iloc[0]['SeedNum']
        return 1.0 / (1.0 + 10.0 ** (diff * 0.15))
    return 0.5

seed_preds = []
for _, row in sub2.iterrows():
    parts = row['ID'].split('_')
    seed_preds.append(seed_prior_func(int(parts[1]), int(parts[2]), int(parts[0])))
seed_preds = np.array(seed_preds)

model_preds = sub2_temporal['Pred'].values
conservative = np.clip(0.20 * seed_preds + 0.80 * model_preds, CLIP_MIN, CLIP_MAX)
sub2_cons = sub2.copy()
sub2_cons['Pred'] = conservative
sub2_cons.to_csv(OUT_DIR / 'submission_stage2_temporal_conservative.csv', index=False)
print(f"  Conservative: mean={conservative.mean():.4f}")

## 9. Final Summary

In [ ]:
print("\n" + "=" * 60)
print("FOUNDATION & TEMPORAL MODELS - FINAL RESULTS")
print("=" * 60)

print("\nModel Performance (Leave-One-Season-Out CV, Brier Score):")
print("-" * 55)
all_scores = {
    'TFT-Lite': bs_tft,
    'BiLSTM-Attn': bs_bilstm,
    'Conv1D': bs_cnn,
    'LR-Temporal': bs_lr_temp,
    'TS-Embedder': bs_tsembed,
}
if HAS_CHRONOS:
    all_scores['Chronos'] = bs_chronos

for name, bs in sorted(all_scores.items(), key=lambda x: x[1]):
    bar = '|' * int((0.25 - bs) * 200)
    print(f"  {name:25s}: {bs:.4f} {bar}")

print(f"\n  {'TEMPORAL ENSEMBLE':25s}: {opt_brier:.4f} ***")
print(f"  {'Simple Average':25s}: {simple_brier:.4f}")
if meta_valid.sum() > 0:
    print(f"  {'Stacking':25s}: {stacking_brier:.4f}")

print(f"\nEnsemble Weights (non-zero):")
for name, w in sorted(opt_weights.items(), key=lambda x: -x[1]):
    if w > 0.01:
        print(f"  {name:25s}: {w:.1%}")

print(f"\nSubmissions generated:")
print(f"  {OUT_DIR / 'submission_stage1_temporal.csv'}")
print(f"  {OUT_DIR / 'submission_stage2_temporal.csv'} (aggressive)")
print(f"  {OUT_DIR / 'submission_stage2_temporal_conservative.csv'} (conservative)")
print(f"\nUpload to Kaggle alongside other notebook submissions for mega-ensemble.")